# Lesson 02: The encoder

In Lesson 01 we built the bricks. Now we stack them into an **encoder**.

**The encoder's job:** take an image and turn it into features at several scales.
Each stage makes H, W **2× smaller** and the number of channels **bigger**.

```
image [3, 352, 352]
   │
 stage1 ──► f1 [ 64, 352, 352]   fine detail: edges, texture        (stride 1)
   │ down
 stage2 ──► f2 [128, 176, 176]                                      (stride 2)
   │ down
 stage3 ──► f3 [256,  88,  88]                                      (stride 4)
   │ down
 stage4 ──► f4 [512,  44,  44]                                      (stride 8)
   │ down
 stage5 ──► f5 [1024, 22,  22]   meaning: "there is a fish here"    (stride 16)
```

**The encoder contract** (remember this, it's the whole lesson):

> `encoder(image)` → **list of feature maps** `[f1, f2, f3, f4, f5]`, from big/shallow to small/deep.

As long as an encoder keeps this contract, you can swap it for a different one (UNet, ResNet50, Res2Net, PVT ...)
and the decoder still works. SINet, UNet, UNet++ and PraNet all follow it.

| Part | Topic |
|---|---|
| A | The original UNet encoder, written out by hand |
| B | A configurable encoder: **the knobs you can modify** |
| C | A pretrained ResNet50 as the encoder (the backbone SINet uses) |
| D | Look inside: what each stage "sees" on a real COD10K image |
| E | GPU budget: memory and speed on your RTX 4050 (6 GB) |

## Setup

In [ ]:
import time

import cv2
import numpy as np
import torch
import torch.nn as nn

from blocks import ConvBNReLU, DepthwiseSeparableConv, DoubleConv, ResidualBlock, count_params, show
from config import IMAGE_SIZE, cod10k_pairs, get_device

device = get_device()
print(f"device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

`blocks.py` holds the bricks from Lesson 01 (`ConvBNReLU`, `DoubleConv`, `ResidualBlock`, `DepthwiseSeparableConv`),
so we can import them here instead of copying them again.

Next we load one COD10K test image and its ground-truth mask. `cod10k_pairs()` in `config.py` returns `(image, mask)` paths.
It keeps only the camouflaged (`-CAM-`) images, as SINet's standard protocol does.

In [ ]:
train_pairs = cod10k_pairs("Train")
test_pairs = cod10k_pairs("Test")
print(f"COD10K camouflaged images -> train: {len(train_pairs)}   test: {len(test_pairs)}   (expected 3040 / 2026)")

SAMPLE_INDEX = 0  # ✏️ change this to look at other images
if test_pairs:
    image_path, mask_path = test_pairs[SAMPLE_INDEX]
    image = cv2.resize(cv2.imread(image_path), (IMAGE_SIZE, IMAGE_SIZE))
    mask = cv2.resize(cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE), (IMAGE_SIZE, IMAGE_SIZE))
    print(image_path)
else:
    print("COD10K not found -> synthetic image (check DATASET_ROOT in config.py)")
    image = np.full((IMAGE_SIZE, IMAGE_SIZE, 3), (60, 120, 80), np.uint8)
    image = cv2.add(image, np.random.default_rng(0).integers(0, 40, image.shape, dtype=np.uint8))
    mask = np.zeros((IMAGE_SIZE, IMAGE_SIZE), np.uint8)
    cv2.circle(image, (176, 176), 70, (70, 135, 90), -1)
    cv2.circle(mask, (176, 176), 70, 255, -1)

show([image, mask], ["image", "ground truth (GT_Object)"], cols=2)

### Image → tensor

Pretrained networks were trained on ImageNet images normalised with a fixed mean/std,
so we always normalise the same way. It does no harm for networks trained from scratch either.

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], np.float32)


def to_tensor(image_bgr):
    """OpenCV BGR uint8 [H, W, 3]  ->  normalised float tensor [1, 3, H, W]."""
    rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
    rgb = (rgb - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(rgb).permute(2, 0, 1).unsqueeze(0)


x = to_tensor(image).to(device)
print(f"input tensor: {tuple(x.shape)}")

## Part A: the original UNet encoder, by hand

Five stages. Each stage is a `DoubleConv`, and between stages a `MaxPool2d(2)` halves H and W.
Channels double every stage: 64 → 128 → 256 → 512 → 1024.

Notice that `forward` **keeps every stage's output** and returns them all as a list.
The decoder will need them (the skip connections).

In [ ]:
class UNetEncoder(nn.Module):
    def __init__(self, in_ch=3):
        super().__init__()
        self.pool = nn.MaxPool2d(2)            # the "down" step: H, W -> H/2, W/2
        self.stage1 = DoubleConv(in_ch, 64)
        self.stage2 = DoubleConv(64, 128)
        self.stage3 = DoubleConv(128, 256)
        self.stage4 = DoubleConv(256, 512)
        self.stage5 = DoubleConv(512, 1024)    # the deepest stage, often called the "bottleneck"

    def forward(self, x):
        f1 = self.stage1(x)                    # [B,   64, 352, 352]
        f2 = self.stage2(self.pool(f1))        # [B,  128, 176, 176]
        f3 = self.stage3(self.pool(f2))        # [B,  256,  88,  88]
        f4 = self.stage4(self.pool(f3))        # [B,  512,  44,  44]
        f5 = self.stage5(self.pool(f4))        # [B, 1024,  22,  22]
        return [f1, f2, f3, f4, f5]


def describe(encoder, x):
    """Print the shape, stride and size of every feature map an encoder returns."""
    encoder.eval()
    with torch.no_grad():
        feats = encoder(x)
    print(f"{'feature':<8}{'shape':<24}{'stride':>7}{'values':>12}")
    for i, f in enumerate(feats, start=1):
        stride = x.shape[-1] // f.shape[-1]
        print(f"f{i:<7}{str(tuple(f.shape)):<24}{stride:>7}{f.numel():>12,}")
    print(f"total params: {count_params(encoder):,}")
    return feats


unet_enc = UNetEncoder().to(device)
unet_feats = describe(unet_enc, x)

In [ ]:
# Where do the parameters live? Almost all of them are in the deep stages.
for name in ["stage1", "stage2", "stage3", "stage4", "stage5"]:
    n = count_params(getattr(unet_enc, name))
    print(f"{name}: {n:>10,} params  {'#' * max(1, n // 300_000)}")

**What to notice**
- The **first** stage has the most *values* (big H×W): it uses the most GPU memory.
- The **last** stage has the most *parameters* (many channels): it takes up most of the model's size.
- `f1` knows *where* edges are but not *what* they belong to. `f5` knows *what* is there but only at 22×22 resolution.
  A decoder's job is to combine the two.

> ✏️ **TRY IT**
> - Feed a different size: `describe(unet_enc, torch.randn(1, 3, 256, 256).to(device))`. Then try 250×250. What goes wrong, and why do people use sizes divisible by 32?

## Part B: a configurable encoder, the knobs you can modify

Instead of hard-coding everything, turn each design choice into an argument.
**These are the parts of an encoder that papers change.**

| knob | options here | effect |
|---|---|---|
| `channels` | e.g. `(64,128,256,512,1024)` or `(32,64,128,256,512)` | width: capacity vs. memory/params |
| `len(channels)` | 4, 5, 6 stages | depth: how small the deepest map gets, how much context it sees |
| `block` | `"double"`, `"residual"`, `"dws"` | what each stage is made of |
| `down` | `"maxpool"`, `"avgpool"`, `"conv"` | how H, W are halved (a strided conv is *learnable*) |

In [ ]:
BLOCKS = {
    "double": DoubleConv,                 # original UNet
    "residual": ResidualBlock,            # ResNet-style, easier to train deep
    "dws": DepthwiseSeparableConv,        # MobileNet-style, very light
}


def make_down(kind, ch):
    if kind == "maxpool":
        return nn.MaxPool2d(2)
    if kind == "avgpool":
        return nn.AvgPool2d(2)
    if kind == "conv":
        return ConvBNReLU(ch, ch, kernel_size=3, stride=2)  # learnable downsampling
    raise ValueError(kind)


class ConfigurableEncoder(nn.Module):
    def __init__(self, in_ch=3, channels=(64, 128, 256, 512, 1024), block="double", down="maxpool"):
        super().__init__()
        Block = BLOCKS[block]
        self.channels = list(channels)
        self.downs = nn.ModuleList()
        self.stages = nn.ModuleList()
        prev = in_ch
        for i, ch in enumerate(channels):
            self.downs.append(nn.Identity() if i == 0 else make_down(down, prev))  # no down before stage 1
            self.stages.append(Block(prev, ch))
            prev = ch

    def forward(self, x):
        feats = []
        for down, stage in zip(self.downs, self.stages):
            x = stage(down(x))
            feats.append(x)
        return feats


# Sanity check: with default settings it should match the hand-written UNetEncoder.
enc = ConfigurableEncoder().to(device)
print(f"ConfigurableEncoder params: {count_params(enc):,}   UNetEncoder params: {count_params(unet_enc):,}")

In [ ]:
VARIANTS = {
    "UNet original":            dict(),
    "UNet light (half width)":  dict(channels=(32, 64, 128, 256, 512)),
    "4 stages":                 dict(channels=(64, 128, 256, 512)),
    "6 stages":                 dict(channels=(32, 64, 128, 256, 512, 1024)),
    "residual blocks":          dict(block="residual"),
    "depthwise-separable":      dict(block="dws"),
    "strided-conv down":        dict(down="conv"),
}

print(f"{'variant':<26}{'params':>12}   {'deepest feature':<22}")
for name, cfg in VARIANTS.items():
    e = ConfigurableEncoder(**cfg).to(device).eval()
    with torch.no_grad():
        deepest = e(x)[-1]
    print(f"{name:<26}{count_params(e):>12,}   {str(tuple(deepest.shape)):<22}")

> ✏️ **TRY IT**
> - Make your own variant, e.g. `dict(channels=(48, 96, 192, 384, 768), block="residual", down="conv")`, and add it to `VARIANTS`.
> - Halving the width cut the params by about 4×, not 2×. Why? (Hint: a conv weight is `out_ch × in_ch × k × k`.)
> - Add a new block type to `BLOCKS`, e.g. a `ConvBNReLU` with `dilation=2`, and see that nothing else needs to change.

## Part C: a pretrained ResNet50 as the encoder

Writing an encoder from scratch means training it from scratch, and COD10K (3040 training images) is small.
So almost every COD paper uses a **backbone pretrained on ImageNet** (1.2M images) as the encoder:

| paper | encoder (backbone) |
|---|---|
| SINet (CVPR 2020) | ResNet50 |
| SINet-V2 (TPAMI 2021) | Res2Net50 |
| newer COD models | PVTv2, Swin, ConvNeXt ... |

torchvision's ResNet50 is a classifier. We keep its feature stages and throw away the classification head (`avgpool`, `fc`).
Then we wrap it so it follows **the same contract** as our encoders.

The first call downloads the weights (~100 MB) to your torch cache. After that it loads offline.

In [ ]:
from torchvision.models import ResNet50_Weights, resnet50


class ResNet50Encoder(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        net = None
        if pretrained:
            try:
                net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
                print("Loaded ImageNet-pretrained ResNet50")
            except Exception as e:  # e.g. no internet
                print(f"Could not download pretrained weights ({type(e).__name__}) -> using random weights")
        if net is None:
            net = resnet50(weights=None)

        self.stem = nn.Sequential(net.conv1, net.bn1, net.relu)  # 7x7 conv, stride 2
        self.pool = net.maxpool                                   # stride 2
        self.layer1, self.layer2 = net.layer1, net.layer2         # layer* are stacks of "Bottleneck" residual blocks
        self.layer3, self.layer4 = net.layer3, net.layer4
        self.channels = [64, 256, 512, 1024, 2048]

    def forward(self, x):
        f1 = self.stem(x)                  # [B,   64, 176, 176]  stride 2
        f2 = self.layer1(self.pool(f1))    # [B,  256,  88,  88]  stride 4
        f3 = self.layer2(f2)               # [B,  512,  44,  44]  stride 8
        f4 = self.layer3(f3)               # [B, 1024,  22,  22]  stride 16
        f5 = self.layer4(f4)               # [B, 2048,  11,  11]  stride 32
        return [f1, f2, f3, f4, f5]


resnet_enc = ResNet50Encoder(pretrained=True).to(device)
resnet_feats = describe(resnet_enc, x)

**Compare with the UNet encoder:**
- ResNet50 shrinks faster: its first feature is already stride 2, and the deepest is stride **32** (11×11), not 16.
- It is 5× deeper (50 layers vs 10) and twice as wide at the end (2048 channels), yet it has only ~25% more params (23.5M vs 18.8M), because its `Bottleneck` blocks do most of their work with cheap 1×1 convs.
- Same contract (a list of 5 features), so a decoder can use either one.

SINet uses `f3, f4, f5` (plus `f2` for detail). That's why the list contract matters: the decoder just picks the scales it wants.

> ✏️ **TRY IT**
> - `print(resnet_enc.layer1[0])` to see one `Bottleneck` block. Find the 1×1 → 3×3 → 1×1 pattern and the `downsample` shortcut. It's the `ResidualBlock` idea from Lesson 01.
> - Count the blocks: `[len(l) for l in (resnet_enc.layer1, resnet_enc.layer2, resnet_enc.layer3, resnet_enc.layer4)]`. 3+4+6+3 = 16 blocks × 3 convs + stem + fc = 50 layers.

## Part D: look inside, what does each stage see?

For every feature map we average over its channels and show the result as a heatmap on top of the image.
- **UNet encoder:** random weights (untrained). It only shows edges and colours.
- **ResNet50:** pretrained on ImageNet. Deeper stages start to light up on *objects*, even camouflaged ones, before any COD training.

In [ ]:
def heatmap_overlay(feature, image_bgr, alpha=0.5):
    """Average a [1, C, h, w] feature over channels -> colour heatmap on top of the image."""
    fmap = feature[0].float().mean(0).cpu().numpy()
    fmap = (fmap - fmap.min()) / (fmap.max() - fmap.min() + 1e-6)
    fmap = cv2.resize(fmap, (image_bgr.shape[1], image_bgr.shape[0]), interpolation=cv2.INTER_LINEAR)
    heat = cv2.applyColorMap((fmap * 255).astype(np.uint8), cv2.COLORMAP_JET)
    return cv2.addWeighted(image_bgr, 1 - alpha, heat, alpha, 0)


def feature_titles(feats):
    return [f"f{i} {f.shape[1]}ch {f.shape[2]}x{f.shape[3]}" for i, f in enumerate(feats, start=1)]


print("UNet encoder (random weights)")
show([heatmap_overlay(f, image) for f in unet_feats] + [mask], feature_titles(unet_feats) + ["GT"], cols=6, size=3)

print("ResNet50 encoder (ImageNet-pretrained)")
show([heatmap_overlay(f, image) for f in resnet_feats] + [mask], feature_titles(resnet_feats) + ["GT"], cols=6, size=3)

The channel average is a crude view: each channel is its own detector, and averaging mixes them all together.
Still, you should see the pretrained deep features (`f4`, `f5`) are **blurry but about objects**, and the shallow ones (`f1`, `f2`) are **sharp but about edges and texture**.
That is exactly why a decoder needs both.

> ✏️ **TRY IT**
> - Change `SAMPLE_INDEX` in the setup cell, re-run the cells from there down, and look at other animals.
> - Instead of the mean, show a single channel: replace `feature[0].float().mean(0)` with `feature[0, 10]`. Try a few channel numbers.

## Part E: GPU budget (RTX 4050 Laptop, 6 GB)

Training stores every intermediate feature for the backward pass, so **memory grows with batch size × image size × width**.
Here we run one forward + backward pass per encoder and measure the peak GPU memory and the time.
`torch.autocast` (mixed precision, float16) roughly halves the memory, and you will use it for training.

In [ ]:
def benchmark(encoder, batch_size, amp=True, size=IMAGE_SIZE):
    encoder.train()
    inp = torch.randn(batch_size, 3, size, size, device=device)
    if device.type == "cuda":
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.synchronize()
    start = time.perf_counter()
    try:
        with torch.autocast(device_type=device.type, dtype=torch.float16 if device.type == "cuda" else torch.bfloat16,
                            enabled=amp):
            feats = encoder(inp)
            loss = sum(f.float().mean() for f in feats)
        loss.backward()
    except torch.OutOfMemoryError:
        encoder.zero_grad(set_to_none=True)
        return "out of memory"
    if device.type == "cuda":
        torch.cuda.synchronize()
    elapsed = (time.perf_counter() - start) * 1000
    encoder.zero_grad(set_to_none=True)
    mem = f"{torch.cuda.max_memory_allocated() / 1024**3:5.2f} GB" if device.type == "cuda" else "  (CPU)"
    return f"{mem}  {elapsed:7.0f} ms"


BATCH_SIZES = [1, 4, 8] if device.type == "cuda" else [1]
encoders = {
    "UNet encoder": unet_enc,
    "UNet light": ConfigurableEncoder(channels=(32, 64, 128, 256, 512)).to(device),
    "ResNet50 encoder": resnet_enc,
}
for name, e in encoders.items():
    benchmark(e, 1)  # warm-up (the first call is always slow)
    for bs in BATCH_SIZES:
        print(f"{name:<18} batch {bs}: {benchmark(e, bs)}")

**How to read this:** the whole model (encoder + decoder) plus the optimizer must fit in 6 GB,
and the decoder adds roughly as much again. A pretrained ResNet50 encoder at 352×352 with AMP and batch 8 is a typical COD setup, and it should fit your GPU.

> ✏️ **TRY IT**
> - Run `benchmark(unet_enc, 4, amp=False)`. How much memory does float16 save?
> - Find the largest batch size that fits for each encoder.

---
## Summary

- An encoder is a stack of **stages**. Each stage is **down → block**, and it returns **one feature per stage**.
- The parts you can modify: **width** (`channels`), **depth** (number of stages), **block type**, **downsampling method**, or the whole thing, by swapping in a **pretrained backbone**.
- Keep the contract (image → list of features) and the rest of the network doesn't care which encoder you use.

**Next lesson (03): the decoder.** Upsampling (`ConvTranspose2d` vs bilinear), and how skip connections merge encoder features back in (concat vs add).